In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
missing_list = pd.read_csv("missing_books.csv", delimiter='\t')
len(missing_list)

101

In [3]:
missing_finalists = pd.read_csv('missing_book_details.csv', delimiter='\t')
missing_finalists

# we see that there are duplicate editions and 

,batch_num,isbn,title,author,release_year,publisher,description,tags
0,3,9.781430e+12,Eifelheim,Michael Flynn,2006,Macmillan,**The alien world of medieval Europe lives aga...,"[{'Science Fiction': 2}, {'Young Adult': 1}, {..."
1,3,9.780765e+12,Eifelheim,Michael Flynn,2006,Macmillan,**The alien world of medieval Europe lives aga...,"[{'Science Fiction': 2}, {'Young Adult': 1}, {..."
2,4,9.788072e+12,Drak Jeho Veličenstva,Naomi Novik,2005,HarperCollins UK,Aerial combat brings a thrilling new dimension...,"[{'Fantasy': 3}, {'Fiction': 2}, {'Science Fic..."
3,4,NaN,His Majesty's Dragon,Naomi Novik,2005,Random House Audio,Aerial combat brings a thrilling new dimension...,"[{'Fantasy': 3}, {'Fiction': 2}, {'Science Fic..."
4,4,9.780739e+12,His Majesty's Dragon,Naomi Novik,2005,RH Audio,Aerial combat brings a thrilling new dimension...,"[{'Fantasy': 3}, {'Fiction': 2}, {'Science Fic..."
...,...,...,...,...,...,...,...,...
356,100,9.781251e+12,The Tomb of Dragons,Katherine Addison,2025,Tor Books,Thara Celehar has lost his ability to speak wi...,"[{'Fantasy': 4}, {'Fiction': 2}, {'Science Fic..."
357,100,9.781838e+12,The Tomb of Dragons,Katherine Addison,2025,Solaris,Thara Celehar has lost his ability to speak wi...,"[{'Fantasy': 4}, {'Fiction': 2}, {'Science Fic..."
358,100,9.781251e+12,The Tomb of Dragons,Katherine Addison,2025,Tor Books,Thara Celehar has lost his ability to speak wi...,"[{'Fantasy': 4}, {'Fiction': 2}, {'Science Fic..."
359,100,9.781251e+12,The Tomb of Dragons,Katherine Addison,2025,Tor Books,Thara Celehar has lost his ability to speak wi...,"[{'Fantasy': 4}, {'Fiction': 2}, {'Science Fic..."


In [4]:
locus = pd.read_csv('locus_nominees.csv')
hugo = pd.read_csv('hugo_nominees.csv')
hugo_mask = missing_finalists.author.isin(hugo.author) & missing_finalists.title.isin(hugo.title)
locus_mask = missing_finalists.author.isin(locus.author) & missing_finalists.title.isin(locus.title)

missing_finalists['hugo'] = hugo_mask
missing_finalists['locus'] = locus_mask

In [5]:
# get rid of any accidental non-matches

missing_finalists = missing_finalists[~((missing_finalists['hugo'] == False) & (missing_finalists['locus'] == False))]

In [6]:
# de-duplicate editions, taking the first published 
missing_finalists = missing_finalists.drop_duplicates(['title', 'author'], keep='first')

In [7]:
missing_finalists.head(5)

,batch_num,isbn,title,author,release_year,publisher,description,tags,hugo,locus
0,3,9.781430e+12,Eifelheim,Michael Flynn,2006,Macmillan,**The alien world of medieval Europe lives aga...,"[{'Science Fiction': 2}, {'Young Adult': 1}, {...",True,True
3,4,NaN,His Majesty's Dragon,Naomi Novik,2005,Random House Audio,Aerial combat brings a thrilling new dimension...,"[{'Fantasy': 3}, {'Fiction': 2}, {'Science Fic...",True,False
22,8,NaN,The Three-Body Problem,Cixin Liu,2006,Head of Zeus,An alien civilization on the brink of destruct...,"[{'Science Fiction': 17}, {'Fiction': 7}, {'Al...",True,True
47,9,9.781430e+12,The Goblin Emperor,Katherine Addison,2014,Tor Books,"The youngest, half-goblin son of the Emperor h...","[{'Fantasy': 8}, {'Fiction': 2}, {'Science Fic...",True,True
57,10,NaN,Death's End,Cixin Liu,2008,Tor Books,"Half a century after the Doomsday Battle, the ...","[{'Science Fiction': 11}, {'Fiction': 3}, {'Fa...",True,True


In [8]:
import ast 
def get_tags(b):
    if pd.isna(b):
        return np.nan
    b_dict = ast.literal_eval(b)
    b_keys =  [list(item.keys())[0] for item in b_dict]
    return b_keys
    

missing_finalists['tags'] = missing_finalists['tags'].apply(get_tags)

In [9]:
missing_finalists['first_publisher'] = missing_finalists['publisher']
missing_finalists['isbn'].apply(lambda x: np.nan if np.isnan(x) else str(x) )
missing_finalists['release_year'].apply(lambda x: np.nan if np.isnan(x) else int(x) )
missing_finalists['book_synopsis'] = missing_finalists['description']


In [10]:
missing_finalists = missing_finalists.drop(['batch_num', 'description', 'publisher'], axis=1)

In [11]:
missing_finalists

,isbn,title,author,release_year,tags,hugo,locus,first_publisher,book_synopsis
0,9.781430e+12,Eifelheim,Michael Flynn,2006,"[Science Fiction, Young Adult, Fiction, Fantas...",True,True,Macmillan,**The alien world of medieval Europe lives aga...
3,NaN,His Majesty's Dragon,Naomi Novik,2005,"[Fantasy, Fiction, Science Fiction, War, Young...",True,False,Random House Audio,Aerial combat brings a thrilling new dimension...
22,NaN,The Three-Body Problem,Cixin Liu,2006,"[Science Fiction, Fiction, Aliens, Fantasy, Sc...",True,True,Head of Zeus,An alien civilization on the brink of destruct...
47,9.781430e+12,The Goblin Emperor,Katherine Addison,2014,"[Fantasy, Fiction, Science Fiction & Fantasy, ...",True,True,Tor Books,"The youngest, half-goblin son of the Emperor h..."
57,NaN,Death's End,Cixin Liu,2008,"[Science Fiction, Fiction, Fantasy, Dystopian,...",True,True,Tor Books,"Half a century after the Doomsday Battle, the ..."
72,9.781789e+12,Nettle & Bone,T. Kingfisher,2022,"[Fantasy, Fiction, Horror, Young Adult, Advent...",True,True,Titan Books,After years of seeing her sisters suffer at th...
82,NaN,A Sorceress Comes to Call,T. Kingfisher,2024,"[Fantasy, Fiction, Science Fiction, Young Adul...",True,True,Macmillan Audio,*The hardcover edition features a foil stamp o...
92,9.780312e+12,All My Sins Remembered,Joe Haldeman,1977,"[Science Fiction, Fantasy, Space, Aliens, Adve...",False,True,St. Martin's Press,Otto McGavin is peaceful and idealistic by nat...
95,9.780553e+12,Tales of Nevèrÿon,Samuel R. Delany,1979,"[Fiction, Fantasy, Classics, Science fiction, ...",False,True,Bantam Books,A group of interrelated stories taking place i...
101,9.780709e+12,King David's Spaceship,Jerry Pournelle,1973,"[Adventure, Fiction, Science Fiction, Space, ...",False,True,Futura,Four levels of technology collide in this fasc...


In [12]:
previous_dataset = pd.read_csv('isfdb-hardcover-hugo.csv')

In [13]:
full_dataset = pd.concat([previous_dataset, missing_finalists], ignore_index=True)

In [14]:
# get rid of old index columns

full_dataset = full_dataset.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis=1)

In [15]:
full_dataset.columns

Index(['title_id', 'author', 'release_year', 'release_date',
       'author_age_at_release', 'author_birthplace', 'isbn', 'first_publisher',
       'book_synopsis', 'tags', 'title', 'hugo', 'locus'],
      dtype='str')

In [16]:
# reordering columns a bit
full_dataset = full_dataset[['title_id', 'title', 'author', 'release_year', 'release_date', 'first_publisher',
       'author_age_at_release', 'author_birthplace', 'isbn', 
       'book_synopsis', 'tags',  'hugo', 'locus']]

In [18]:
full_dataset.to_csv("final_book_dataset.csv", sep='\t')